In [4]:
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

@dataclass(frozen=True)
class Provider:
    name: str
    env_var: str
    base_url: str
    is_free: bool
    model: str

PROVIDERS = [
    Provider("OpenAI", "OPENAI_API_KEY", "https://api.openai.com/v1", False, "gpt-4o-mini"),
    Provider("Grok", "GROK_API_KEY", "https://api.groq.com/openai/v1", True, "openai/gpt-oss-20b"),
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider
    expected = ", ".join([p.env_var for p in PROVIDERS])
    raise RuntimeError(f"None of the expected environment variables are set. Expected one of: {expected}")


def build_client(provider: Provider):
    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)
    return OpenAI(api_key=api_key, base_url=provider.base_url)


def have_any_keys() -> bool:
    return any(os.getenv(provider.env_var) for provider in PROVIDERS)

print(
    "Found a provider with an API key."
    if have_any_keys()
    else "No providers found with an API key."
)

# earlier we were having client.chat.completion.create, but now we are using client.responses.create
# in completions we were using messages, but in responses we are using input
# in completrions we need to define role, but in responses we don't need to define role

def llm_response(prompt: str) -> str:
    provider = select_provider()
    print(f"Using provider: {provider.name} with model: {provider.model}")
    client = build_client(provider)
    response = client.responses.create(
        model=provider.model,
        input=prompt
    )
    return response.output_text

prompt = "Who is the president of the United States?"
response = llm_response(prompt)
print(f"Prompt: {prompt}")
print(f"Response: {response}")

Found a provider with an API key.
Using provider: Grok with model: openai/gpt-oss-20b
Prompt: Who is the president of the United States?
Response: As of my last update (June 2024), the President of the United States is **Joe Biden**.
